# EvaLoop Assignment 4 (part 3): resume Seed-Coder-8B, code↔pseudocode loop

The first session hit Kaggle's 12 h limit during this run. This notebook continues it from the per-cycle checkpoint,
running only the remaining cycles for tasks that were still in the loop.

**Before running:** go to **Add Input → Upload → New Dataset** and upload `seed-coder-8b-instruct__code_pseudocode.checkpoint.json`
(from the first session's Output → `EvaLooop/results/`). Settings: GPU T4 x2, Internet On, then Save & Run All.

In [ ]:
import os, glob, json, shutil, subprocess, time
FORK_URL = "https://github.com/Arn15/EvaLooop.git"
MODEL = "ByteDance-Seed/Seed-Coder-8B-Instruct"
LOOP = "code_pseudocode"
# Same engine settings as the first Seed-Coder session (default attention backend, 0.90 memory),
# so the resumed cycles are generated exactly like the earlier ones.
EXTRA_ARGS = "--gpu_memory_utilization 0.90"
ckpts = glob.glob('/kaggle/input/**/seed-coder-8b-instruct__code_pseudocode.checkpoint.json', recursive=True)
assert ckpts, "upload the checkpoint JSON as a Kaggle dataset first (Add Input)"
ck = json.load(open(ckpts[0]))
alive = sum(1 for r in ck["result"]["prompt_results"] if r["successful_cycles"] == ck["completed_cycle"])
print("checkpoint:", ckpts[0], "| completed cycles:", ck["completed_cycle"], "| tasks still in the loop:", alive)

In [ ]:
os.chdir('/kaggle/working')
if not os.path.exists('EvaLooop'):
    !git clone -q -b main {FORK_URL} EvaLooop
os.chdir('/kaggle/working/EvaLooop')
!git log --oneline | head -2
os.makedirs('results', exist_ok=True)
shutil.copy(ckpts[0], 'results/resume_from.checkpoint.json')

In [ ]:
!pip uninstall -y -q torchaudio
!pip install -q "vllm==0.10.2" "transformers==4.57.6" "huggingface-hub<1.0" "evalplus==0.3.1" fire tqdm matplotlib 2>&1 | grep -v "requires\|incompatible" | tail -3
!python -c "import torch, vllm, transformers; print('transformers', transformers.__version__, '| torch', torch.__version__, '| vllm', vllm.__version__)"
!python -W ignore -c "import sys; sys.path.insert(0,'.'); from evaloop.evaluation.py_testing import load_mbpp_dataset; p,_=load_mbpp_dataset(); print(len(p),'tasks')"
!python -m pytest -q tests 2>&1 | tail -2

In [ ]:
os.makedirs('logs', exist_ok=True)
t0 = time.time()
cmd = (f"python scripts/run_loop.py --model {MODEL} --loop {LOOP} --backend vllm_chat --dtype half "
       f"--tensor_parallel_size 2 --max_model_len 8192 {EXTRA_ARGS} "
       f"--resume results/resume_from.checkpoint.json --output_dir results/ --log_level WARNING")
print('>>>', cmd, flush=True)
with open("logs/seed-coder-8b-instruct__code_pseudocode_resume.log", "w") as log:
    rc = subprocess.run(cmd, shell=True, stdout=log, stderr=subprocess.STDOUT).returncode
print(open("logs/seed-coder-8b-instruct__code_pseudocode_resume.log").read()[-1500:])
print(f"exit={rc}  minutes={(time.time()-t0)/60:.1f}")
assert rc == 0

In [ ]:
!cd /kaggle/working/EvaLooop && zip -qr /kaggle/working/evaloop_results_seed_resume.zip results logs
!ls -la /kaggle/working/evaloop_results_seed_resume.zip